# ResNet-MLP
        **tabular › deep_regression** · notebook `resnet`

        **Kapan dipakai:** Varian MLP dengan koneksi residual dan BatchNorm; lebih stabil saat lebih dalam.

        **Jebakan umum:** Sama seperti MLP; sensitif learning rate.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "resnet"
TASK       = "regression"
KIND       = "nn"
OUT = out_dir("tabular", "deep_regression", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model deep (PyTorch)
Diberi DataFrame mentah: numerik distandarkan, kategori diberi embedding, berhenti dini pada 10% data validasi internal. GPU dipakai otomatis.

In [ ]:
from superdb.deep_tabular import TabNN

model = TabNN(arch='resnet', task=TASK, num_cols=NUM, cat_cols=CAT, hidden=128, depth=3, dropout=0.2, epochs=80)
import time
from sklearn.base import clone
from superdb.deep_tabular import TabNN
pipe = model

## 5. Cross-validation vs LightGBM (lipatan yang sama)
Pembanding wajib: pada data tabular, boosting masih menjadi tolok ukur.

In [ ]:
from lightgbm import LGBMClassifier, LGBMRegressor
t0 = time.time(); folds, summ = cv_score(model, Xtr, ytr, TASK, SCORING, 3, SEED); t_nn = time.time() - t0
lg = (LGBMClassifier if TASK == "classification" else LGBMRegressor)(n_estimators=300, learning_rate=0.05, num_leaves=15, random_state=SEED, n_jobs=-1, verbose=-1)
t0 = time.time(); _, summ_lgb = cv_score(build_pipeline(lg, NUM, CAT, kind="tree"), Xtr, ytr, TASK, SCORING, 3, SEED); t_lg = time.time() - t0
display(pd.DataFrame({"deep (" + MODEL_KEY + ")": {"cv_mean": summ["valid_mean"], "cv_std": summ["valid_std"], "detik": t_nn}, "LightGBM": {"cv_mean": summ_lgb["valid_mean"], "cv_std": summ_lgb["valid_std"], "detik": t_lg}}).T.round(4))

## 6. Latih & evaluasi pada validation

In [ ]:
model.fit(Xtr, ytr); print("epoch terbaik:", model.best_epoch_)

In [ ]:
pred = pipe.predict(Xva)
metrics = regression_metrics(yva, pred)
display(show_table(metrics))
plot_regression(yva, pred, path=OUT / "regression.png"); plt.show()

## 7. Latih ulang pada semua data → submission → simpan

In [ ]:
final = clone(model).fit(X, y)
if test is not None:
    Xte, _, _ = prepare_features(test, cols); Xte = Xte.reindex(columns=X.columns)
    ids = test[ID_COL] if ID_COL else np.arange(len(test))
    sub = make_submission(ids, final.predict(Xte), ID_COL or "id", TARGET, OUT / "submission.csv"); display(sub.head(3))
save_model({"model": final, "cols": cols}, OUT / "model.joblib")
save_json({"cv": summ, "cv_lightgbm": summ_lgb, "valid": metrics, "best_epoch": int(model.best_epoch_)}, OUT / "metrics.json")
b = load_model(OUT / "model.joblib")["model"]; Xc = Xva.reindex(columns=X.columns).head(50)
assert np.allclose(np.asarray(b.predict(Xc), float), np.asarray(final.predict(Xc), float), atol=1e-4), "model muat-ulang beda"; print("OK. hasil di", OUT)

## Langkah berikutnya
- Jika deep learning tidak mengalahkan LightGBM (umum pada data kecil-menengah), gunakan keduanya dalam **blend**: kesalahannya berbeda.
- Tuning: `hidden`, `depth`, `dropout`, `lr`; data besar mendapat manfaat lebih dari FT-Transformer.